### 1. Loading Datasets

In [49]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

In [ ]:
calls = pd.read_csv("../Data/calls.csv")
customers = pd.read_csv("../Data/customers.csv")
reason = pd.read_csv("../Data/reason.csv")
sentiment = pd.read_csv("../Data/sentiment_statistics.csv")

In [51]:
for name, df in {
    "calls": calls,
    "customers": customers,
    "reason": reason,
    "sentiment": sentiment
}.items():
    print(f"{name.upper()}")
    display(df.head())

CALLS


,call_id,customer_id,agent_id,call_start_datetime,agent_assigned_datetime,call_end_datetime,call_transcript
0,4667960400,2033123310,963118,7/31/2024 23:56,8/1/2024 0:03,8/1/2024 0:34,\n\nAgent: Thank you for calling United Airlin...
1,1122072124,8186702651,519057,8/1/2024 0:03,8/1/2024 0:06,8/1/2024 0:18,\n\nAgent: Thank you for calling United Airlin...
2,6834291559,2416856629,158319,7/31/2024 23:59,8/1/2024 0:07,8/1/2024 0:26,\n\nAgent: Thank you for calling United Airlin...
3,2266439882,1154544516,488324,8/1/2024 0:05,8/1/2024 0:10,8/1/2024 0:17,\n\nAgent: Thank you for calling United Airlin...
4,1211603231,5214456437,721730,8/1/2024 0:04,8/1/2024 0:14,8/1/2024 0:23,\n\nAgent: Thank you for calling United Airlin...


CUSTOMERS


,customer_id,customer_name,elite_level_code
0,2033123310,Matthew Foster,4.000
1,8186702651,Tammy Walters,NaN
2,2416856629,Jeffery Dixon,NaN
3,1154544516,David Wilkins,2.000
4,5214456437,Elizabeth Daniels,0.000


REASON


,call_id,primary_call_reason
0,4667960400,Voluntary Cancel
1,1122072124,Booking
2,6834291559,IRROPS
3,2266439882,Upgrade
4,1211603231,Seating


SENTIMENT


,call_id,agent_id,agent_tone,customer_tone,average_sentiment,silence_percent_average
0,4667960400,963118,neutral,angry,-0.040,0.390
1,1122072124,519057,calm,neutral,0.020,0.350
2,6834291559,158319,neutral,polite,-0.130,0.320
3,2266439882,488324,neutral,frustrated,-0.200,0.200
4,1211603231,721730,neutral,polite,-0.050,0.350


### 2. Cleaning Datasets and Creating New Features

#### calls.csv

In [52]:
calls.head()

,call_id,customer_id,agent_id,call_start_datetime,agent_assigned_datetime,call_end_datetime,call_transcript
0,4667960400,2033123310,963118,7/31/2024 23:56,8/1/2024 0:03,8/1/2024 0:34,\n\nAgent: Thank you for calling United Airlin...
1,1122072124,8186702651,519057,8/1/2024 0:03,8/1/2024 0:06,8/1/2024 0:18,\n\nAgent: Thank you for calling United Airlin...
2,6834291559,2416856629,158319,7/31/2024 23:59,8/1/2024 0:07,8/1/2024 0:26,\n\nAgent: Thank you for calling United Airlin...
3,2266439882,1154544516,488324,8/1/2024 0:05,8/1/2024 0:10,8/1/2024 0:17,\n\nAgent: Thank you for calling United Airlin...
4,1211603231,5214456437,721730,8/1/2024 0:04,8/1/2024 0:14,8/1/2024 0:23,\n\nAgent: Thank you for calling United Airlin...


In [53]:
calls.isnull().any()

call_id                    False
customer_id                False
agent_id                   False
call_start_datetime        False
agent_assigned_datetime    False
call_end_datetime          False
call_transcript            False
dtype: bool

In [54]:
datetime_cols = [
    "call_start_datetime",
    "agent_assigned_datetime",
    "call_end_datetime"
]

for col in datetime_cols:
    calls[col] = pd.to_datetime(calls[col], errors="coerce")

In [55]:
calls[datetime_cols].isnull().any()

call_start_datetime        False
agent_assigned_datetime    False
call_end_datetime          False
dtype: bool

In [56]:
calls = calls.dropna(subset=datetime_cols).copy()

In [57]:
# AST (Average Speed To Answer) (Seconds)
calls["AST_seconds"] = (calls["agent_assigned_datetime"] - calls["call_start_datetime"]).dt.total_seconds()

In [58]:
# AHT (Average Handle Time) (Minutes)
calls["AHT_minutes"] = (calls["call_end_datetime"] - calls["agent_assigned_datetime"]).dt.total_seconds() / 60

In [59]:
# Total Call Duration (Minutes)
calls["call_duration_minutes"] = (calls["call_end_datetime"] - calls["call_start_datetime"]).dt.total_seconds() / 60

In [60]:
calls[["AST_seconds","AHT_minutes","call_duration_minutes"]].describe()

,AST_seconds,AHT_minutes,call_duration_minutes
count,71810.000,71810.000,71810.000
mean,437.068,11.617,18.902
std,151.130,12.905,13.080
min,180.000,0.000,3.000
25%,300.000,4.000,11.000
50%,420.000,7.000,15.000
75%,540.000,15.000,22.000
max,900.000,119.000,129.000


In [61]:
calls = calls[
    (calls["AST_seconds"] >= 0) &
    (calls["AHT_minutes"] > 0) &
    (calls["call_duration_minutes"] > 0)
]
calls = calls.copy()

In [62]:
calls[["AST_seconds","AHT_minutes","call_duration_minutes"]].describe()

,AST_seconds,AHT_minutes,call_duration_minutes
count,70618.000,70618.000,70618.000
mean,436.973,11.814,19.096
std,151.068,12.924,13.099
min,180.000,1.000,4.000
25%,300.000,4.000,11.000
50%,420.000,7.500,15.000
75%,540.000,15.000,22.000
max,900.000,119.000,129.000


In [63]:
calls["call_hour"] = calls["call_start_datetime"].dt.hour
calls["call_day_of_week"] = calls["call_start_datetime"].dt.day_name()
calls["is_weekend"] = calls["call_day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

#### customers.csv

In [64]:
customers.head()

,customer_id,customer_name,elite_level_code
0,2033123310,Matthew Foster,4.000
1,8186702651,Tammy Walters,NaN
2,2416856629,Jeffery Dixon,NaN
3,1154544516,David Wilkins,2.000
4,5214456437,Elizabeth Daniels,0.000


In [65]:
customers.isnull().any()

customer_id         False
customer_name       False
elite_level_code     True
dtype: bool

In [66]:
customers["elite_level_code"] = (customers["elite_level_code"].fillna(0).astype(int))

In [67]:
customers["elite_level_code"].unique()

array([4, 0, 2, 5, 1, 3])

In [68]:
def elite_tier(code):
    if code == 0:
        return "Non-Elite"
    elif code <= 2:
        return "Mid-Elite"
    else:
        return "High-Elite"

customers["elite_tier"] = customers["elite_level_code"].apply(elite_tier)

In [69]:
customers["customer_id"].is_unique

True

#### reason.csv

In [70]:
reason.head()

,call_id,primary_call_reason
0,4667960400,Voluntary Cancel
1,1122072124,Booking
2,6834291559,IRROPS
3,2266439882,Upgrade
4,1211603231,Seating


In [71]:
reason.isnull().any()

call_id                False
primary_call_reason    False
dtype: bool

In [72]:
reason["primary_call_reason"] = (reason["primary_call_reason"].str.replace(r"\s+", " ", regex=True).str.strip().str.title())

In [73]:
top_reasons = reason["primary_call_reason"].value_counts().nlargest(10).index
top_reasons

Index(['Irrops', 'Voluntary Change', 'Seating', 'Mileage Plus', 'Post-Flight',
       'Communications', 'Products And Services', 'Baggage', 'Upgrade',
       'Booking'],
      dtype='object', name='primary_call_reason')

In [74]:
reason["call_reason_grouped"] = reason["primary_call_reason"].apply(
    lambda x: x if x in top_reasons else "Other"
)

In [77]:
reason["call_reason_grouped"].value_counts()


call_reason_grouped
Irrops                   13311
Other                    11418
Voluntary Change         10848
Seating                   6365
Mileage Plus              5851
Post-Flight               3957
Communications            3840
Products And Services     2856
Baggage                   2832
Upgrade                   2738
Booking                   2637
Name: count, dtype: int64

#### sentiment_statistics.csv

In [78]:
sentiment.head()

,call_id,agent_id,agent_tone,customer_tone,average_sentiment,silence_percent_average
0,4667960400,963118,neutral,angry,-0.040,0.390
1,1122072124,519057,calm,neutral,0.020,0.350
2,6834291559,158319,neutral,polite,-0.130,0.320
3,2266439882,488324,neutral,frustrated,-0.200,0.200
4,1211603231,721730,neutral,polite,-0.050,0.350


In [84]:
sentiment.isnull().any()

call_id                    False
agent_id                   False
agent_tone                  True
customer_tone              False
average_sentiment           True
silence_percent_average    False
dtype: bool

In [85]:
sentiment["agent_tone"] = sentiment["agent_tone"].fillna("neutral")
sentiment["average_sentiment"] = sentiment["average_sentiment"].fillna(0)
sentiment.isnull().any()

call_id                    False
agent_id                   False
agent_tone                 False
customer_tone              False
average_sentiment          False
silence_percent_average    False
dtype: bool

In [86]:
sentiment["customer_tone"].unique()

array(['angry', 'neutral', 'polite', 'frustrated', 'calm'], dtype=object)

In [87]:
sentiment["agent_tone"].unique()

array(['neutral', 'calm', 'frustrated', 'angry', 'polite'], dtype=object)

In [88]:
tone_map = {
    "angry": -2,
    "frustrated": -1,
    "neutral": 0,
    "calm": 1,
    "polite": 2
}

sentiment["customer_tone_score"] = sentiment["customer_tone"].map(tone_map)
sentiment["agent_tone_score"] = sentiment["agent_tone"].map(tone_map)

In [89]:
sentiment["negative_customer_flag"] = sentiment["customer_tone_score"] < 0
sentiment["high_silence_flag"] = sentiment["silence_percent_average"] > 0.4

In [95]:
sentiment[["customer_tone_score", "agent_tone_score", "average_sentiment"]].describe()

,customer_tone_score,agent_tone_score,average_sentiment
count,71810.000,71810.000,71810.000
mean,0.000,0.284,-0.033
std,1.415,0.584,0.144
min,-2.000,-2.000,-1.380
25%,-1.000,0.000,-0.110
50%,0.000,0.000,-0.020
75%,1.000,1.000,0.050
max,2.000,2.000,2.670


In [96]:
sentiment.head()

,call_id,agent_id,agent_tone,customer_tone,average_sentiment,silence_percent_average,customer_tone_score,agent_tone_score,negative_customer_flag,high_silence_flag
0,4667960400,963118,neutral,angry,-0.040,0.390,-2,0,True,False
1,1122072124,519057,calm,neutral,0.020,0.350,0,1,False,False
2,6834291559,158319,neutral,polite,-0.130,0.320,2,0,False,False
3,2266439882,488324,neutral,frustrated,-0.200,0.200,-1,0,True,False
4,1211603231,721730,neutral,polite,-0.050,0.350,2,0,False,False


#### test.csv

In [90]:
test.head()

,call_id
0,7732610078
1,2400299738
2,6533095063
3,7774450920
4,9214147168


In [91]:
test.isnull().any()

call_id    False
dtype: bool

In [94]:
test.duplicated().sum()

np.int64(0)

### 3. Creating the Master Dataset

In [97]:
df = (
    calls
    .merge(customers, on="customer_id", how="left")
    .merge(reason[["call_id", "call_reason_grouped"]], on="call_id", how="left")
    .merge(sentiment, on="call_id", how="left")
)

In [98]:
df.head()

,call_id,customer_id,agent_id_x,call_start_datetime,agent_assigned_datetime,call_end_datetime,call_transcript,AST_seconds,AHT_minutes,call_duration_minutes,call_hour,call_day_of_week,is_weekend,customer_name,elite_level_code,elite_tier,call_reason_grouped,agent_id_y,agent_tone,customer_tone,average_sentiment,silence_percent_average,customer_tone_score,agent_tone_score,negative_customer_flag,high_silence_flag
0,4667960400,2033123310,963118,2024-07-31 23:56:00,2024-08-01 00:03:00,2024-08-01 00:34:00,\n\nAgent: Thank you for calling United Airlin...,420.000,31.000,38.000,23,Wednesday,0,Matthew Foster,4,High-Elite,Other,963118,neutral,angry,-0.040,0.390,-2,0,True,False
1,1122072124,8186702651,519057,2024-08-01 00:03:00,2024-08-01 00:06:00,2024-08-01 00:18:00,\n\nAgent: Thank you for calling United Airlin...,180.000,12.000,15.000,0,Thursday,0,Tammy Walters,0,Non-Elite,Booking,519057,calm,neutral,0.020,0.350,0,1,False,False
2,6834291559,2416856629,158319,2024-07-31 23:59:00,2024-08-01 00:07:00,2024-08-01 00:26:00,\n\nAgent: Thank you for calling United Airlin...,480.000,19.000,27.000,23,Wednesday,0,Jeffery Dixon,0,Non-Elite,Irrops,158319,neutral,polite,-0.130,0.320,2,0,False,False
3,2266439882,1154544516,488324,2024-08-01 00:05:00,2024-08-01 00:10:00,2024-08-01 00:17:00,\n\nAgent: Thank you for calling United Airlin...,300.000,7.000,12.000,0,Thursday,0,David Wilkins,2,Mid-Elite,Upgrade,488324,neutral,frustrated,-0.200,0.200,-1,0,True,False
4,1211603231,5214456437,721730,2024-08-01 00:04:00,2024-08-01 00:14:00,2024-08-01 00:23:00,\n\nAgent: Thank you for calling United Airlin...,600.000,9.000,19.000,0,Thursday,0,Elizabeth Daniels,0,Non-Elite,Seating,721730,neutral,polite,-0.050,0.350,2,0,False,False


In [99]:
df.isnull().any()

call_id                    False
customer_id                False
agent_id_x                 False
call_start_datetime        False
agent_assigned_datetime    False
call_end_datetime          False
call_transcript            False
AST_seconds                False
AHT_minutes                False
call_duration_minutes      False
call_hour                  False
call_day_of_week           False
is_weekend                 False
customer_name              False
elite_level_code           False
elite_tier                 False
call_reason_grouped         True
agent_id_y                 False
agent_tone                 False
customer_tone              False
average_sentiment          False
silence_percent_average    False
customer_tone_score        False
agent_tone_score           False
negative_customer_flag     False
high_silence_flag          False
dtype: bool

In [100]:
df["call_reason_grouped"] = df["call_reason_grouped"].fillna("Other")

In [101]:
df.shape

(70618, 26)

In [102]:
df.to_csv("../Outputs/master_call_center_data.csv", index=False)